# RETECO Track 1a — Full 13-Domain Benchmark

**EN** — Runs the full Track 1a pipeline (BM25 / dense / weighted hybrid / hybrid+cross-encoder rerank) over all 13 Track 1 domains. Designed to survive Kaggle session limits: results are written after every domain and every run is resumable with `--skip-existing`.

**VI** — Chạy pipeline Track 1a đầy đủ (BM25 / dense / weighted hybrid / hybrid+rerank) trên cả 13 domain Track 1. Thiết kế để sống sót qua giới hạn session Kaggle: kết quả ghi lại sau **mỗi** domain, và mọi lần chạy đều tiếp tục được bằng `--skip-existing`.

---

### Before you start / Trước khi bắt đầu
- **GPU** on (right panel → Accelerator → GPU T4 x2) · **Internet** on.
  Bật **GPU** và **Internet** ở panel phải.
- Push the latest code to GitHub first — this notebook clones it, it does not read a Kaggle Dataset.
  Push code mới nhất lên GitHub trước — notebook này clone từ đó, không đọc Kaggle Dataset.

### Budget / Ngân sách
- Download (Track 1 only): **~4.4 GB** · Tải về (chỉ Track 1): ~4.4 GB
- Caches on disk: **~4 GB** (embeddings fp16 + BM25 indexes) · Cache trên đĩa: ~4 GB
- `/kaggle/working` limit is 19.5 GB — the above fits, with room to spare.
  Giới hạn `/kaggle/working` là 19.5 GB — vừa đủ thoải mái.
- GPU: roughly **2–3 h** for the first full pass (encoding dominates). Free quota is 30 h/week.
  GPU: khoảng **2–3 giờ** cho lượt chạy đầy đủ đầu tiên (chủ yếu là encode). Quota free 30 giờ/tuần.


## 1. Clone code from GitHub / Lấy code từ GitHub

**EN** — GitHub is the single source of truth for code. Nothing is uploaded as a dataset, so there is never a question of whether the newest version is in use.

**VI** — GitHub là nguồn code duy nhất. Không upload dataset nữa, nên không bao giờ phải băn khoăn "đang chạy bản mới nhất chưa".


In [ ]:
import os, shutil, torch

GITHUB_REPO = "https://github.com/peotrannnn/RETECO.git"

ROOT = "/kaggle/working/RETECO-project"
os.makedirs(ROOT, exist_ok=True)
for sub in ["reteco_data", "cache/embeddings", "runs/track1a", "runs/scratch"]:
    os.makedirs(f"{ROOT}/{sub}", exist_ok=True)

# Refresh code only -- data, caches and results are left untouched so this
# cell is safe to re-run mid-experiment.
# Chi lam moi code -- data, cache va ket qua khong bi dung toi, nen chay lai
# cell nay giua chung thi nghiem van an toan.
if os.path.isdir(f"{ROOT}/_repo"):
    shutil.rmtree(f"{ROOT}/_repo")
!git clone -q {GITHUB_REPO} {ROOT}/_repo
if os.path.isdir(f"{ROOT}/src"):
    shutil.rmtree(f"{ROOT}/src")
shutil.copytree(f"{ROOT}/_repo/src", f"{ROOT}/src")
shutil.rmtree(f"{ROOT}/_repo")

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE -- enable it in the right panel!")
print("src/track1a:", sorted(os.listdir(f"{ROOT}/src/track1a")))


## 2. (Optional) Hugging Face token / Token Hugging Face
**EN** — Avoids the unauthenticated rate-limit warning. Skip if not set up.
**VI** — Tránh cảnh báo rate-limit. Bỏ qua nếu chưa thiết lập.


In [ ]:
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN set")
except Exception as e:
    print("Skipping HF token / Bo qua:", e)


## 3. Dependencies & organizer scorer / Dependency & scorer của ban tổ chức


In [ ]:
%pip install -q huggingface_hub sentence-transformers faiss-cpu

if not os.path.isdir(f"{ROOT}/RETECO"):
    !git clone -q https://github.com/DataScienceUIBK/RETECO.git {ROOT}/RETECO
print("starter kit:", os.path.isfile(f"{ROOT}/RETECO/starter_kit/scorer.py"))


## 4. Config & data download / Cấu hình & tải dữ liệu

**EN** — All 13 Track 1 domains, declared in one place. Only `track1_tempo/` is downloaded — Track 2 is a different task and would waste ~1 GB. `snapshot_download` resumes, so re-running this cell after an interrupted download continues rather than restarting.

**VI** — Cả 13 domain Track 1, khai báo ở một chỗ duy nhất. Chỉ tải `track1_tempo/` — Track 2 là task khác, tải về chỉ phí ~1 GB. `snapshot_download` có khả năng tiếp tục, nên chạy lại cell này sau khi tải dở sẽ đi tiếp chứ không tải lại từ đầu.


In [ ]:
ALL_DOMAINS = [
    "bitcoin", "cardano", "economics", "genealogy", "history", "hsm", "iota",
    "law", "monero", "politics", "quant", "travel", "workplace",
]
SPLIT = "train"          # keep "dev" for the single final held-out check

from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="DataScience-UIBK/RETECO-SemEval2027",
    repo_type="dataset",
    local_dir=f"{ROOT}/reteco_data",
    allow_patterns=["track1_tempo/*", "split_manifest.json"],
)

!du -sh {ROOT}/reteco_data
print(f"{len(ALL_DOMAINS)} domains ready")


## 5. Main run — order matters / Lượt chạy chính — thứ tự quan trọng

**EN** — `hybrid` runs **first on purpose**: it builds *both* the BM25 index cache and the embedding cache for every domain. The other three pipelines then reuse those caches, so they cost almost nothing beyond their own extra stage. Running `bm25` first instead would leave the expensive encoding still to do.

Every command carries `--skip-existing`, so if the session dies you re-run this same cell and it picks up where it stopped.

**VI** — `hybrid` chạy **trước là có chủ đích**: nó dựng *cả* BM25 index cache lẫn embedding cache cho mọi domain. Ba pipeline còn lại dùng lại cache đó nên gần như không tốn thêm gì ngoài phần việc riêng của chúng. Nếu chạy `bm25` trước thì phần encode đắt đỏ vẫn còn nguyên đó.

Mọi lệnh đều có `--skip-existing`, nên session chết thì chỉ cần chạy lại đúng cell này, nó sẽ đi tiếp từ chỗ dừng.


In [ ]:
import time

PIPELINES = [
    ("hybrid", ["--method", "hybrid"]),                        # builds both caches
    ("bm25",   ["--method", "bm25"]),                          # cached index -> fast
    ("dense",  ["--method", "dense"]),                         # cached embeddings -> fast
    ("hybrid_rerank", ["--method", "hybrid", "--rerank"]),     # + cross-encoder
]

for name, flags in PIPELINES:
    print(f"\n{'='*70}\n### {name}\n{'='*70}", flush=True)
    t0 = time.time()
    cmd = " ".join(flags)
    !cd {ROOT} && python src/track1a/run_release.py --split {SPLIT} {cmd} --skip-existing
    print(f"### {name} finished in {(time.time()-t0)/60:.1f} min", flush=True)


## 6. Results / Kết quả

**EN** — Macro first, then the per-domain breakdown. The per-domain view is what tells you *where* the pipeline is weak — the macro average alone hides a domain scoring near zero.

**VI** — Macro trước, rồi chi tiết từng domain. Bảng theo domain mới cho biết pipeline **yếu ở đâu** — chỉ nhìn macro thì một domain gần 0 điểm sẽ bị che mất.


In [ ]:
import json, glob

summaries = {}
for f in sorted(glob.glob(f"{ROOT}/runs/track1a/results_{SPLIT}_*.json")):
    r = json.load(open(f))
    summaries[r["pipeline"]] = r

print(f"MACRO nDCG@10 -- split={SPLIT}")
print("-" * 52)
for name, r in sorted(summaries.items(), key=lambda kv: -kv[1]["macro_nDCG@10"]):
    print(f"  {name:16s} {r['macro_nDCG@10']:.4f}   ({r['num_domains_recorded']}/13 domains)")
print("\n  reference: official BM25 baseline, full set = 0.0879 (train)")


In [ ]:
# Per-domain breakdown / Chi tiet tung domain
names = list(summaries)
print(f"{'domain':<12}" + "".join(f"{n:>16}" for n in names))
print("-" * (12 + 16 * len(names)))
for d in ALL_DOMAINS:
    row = f"{d:<12}"
    for n in names:
        m = summaries[n]["per_domain"].get(d)
        row += f"{m['metrics']['nDCG@10']:>16.4f}" if m else f"{'-':>16}"
    print(row)


## 7. Package results / Đóng gói kết quả

**EN** — Only `runs/track1a` (official). `runs/scratch` and the caches are deliberately excluded.
**VI** — Chỉ `runs/track1a` (chính thức). `runs/scratch` và cache cố tình không đưa vào.


In [ ]:
!cd {ROOT} && zip -qr /kaggle/working/runs_track1a_output.zip runs/track1a
!ls -lh /kaggle/working/runs_track1a_output.zip
print("\nNow: Save Version (Commit), then pull it down with:")
print("  kaggle kernels output giabotrnl/<kernel-slug> -p D:\\RETECO-project\\kaggle_download")


---
## If the session dies / Nếu session chết giữa chừng

**EN** — Nothing finished is lost: the summary files are written after every domain. To resume:
1. Re-run cells 1–4 (clone, deps, data — `snapshot_download` resumes the download).
2. Re-run cell 5 unchanged. Domains already recorded are skipped.

The one thing that *is* lost is `cache/` if `/kaggle/working` was wiped, so already-encoded domains get re-encoded. To avoid that on a long run, Save Version before the session's 12-hour limit.

**VI** — Không mất gì đã xong: file summary được ghi sau mỗi domain. Để tiếp tục:
1. Chạy lại cell 1–4 (clone, deps, data — `snapshot_download` tự tiếp tục tải).
2. Chạy lại cell 5 y nguyên. Domain đã ghi nhận sẽ được bỏ qua.

Thứ **sẽ** mất là `cache/` nếu `/kaggle/working` bị xoá, khiến domain đã encode phải encode lại. Để tránh, hãy Save Version trước khi chạm giới hạn 12 giờ của session.

## After the run / Sau khi chạy xong

**EN**
1. Compare the four pipelines. Expect `hybrid_rerank` to lead — if it does not, say so, that is a real finding worth reporting.
2. Sweep `--sparse-weight` (0.0 / 0.15 / 0.3 / 0.6) on `runs/scratch` to tune the fusion, now on 13 domains rather than 7 questions.
3. Only then run `--split dev` once, as the held-out check.

**VI**
1. So sánh 4 pipeline. Kỳ vọng `hybrid_rerank` dẫn đầu — nếu không, cứ ghi nhận đúng như vậy, đó là phát hiện thật đáng báo cáo.
2. Sweep `--sparse-weight` (0.0 / 0.15 / 0.3 / 0.6) vào `runs/scratch` để tinh chỉnh fusion, lần này trên 13 domain chứ không phải 7 câu hỏi.
3. Xong hết mới chạy `--split dev` một lần duy nhất, làm bước kiểm tra giữ lại.
